In [1]:
# Importing necessary libraries

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, brier_score_loss, RocCurveDisplay
from sklearn.model_selection import TimeSeriesSplit
import joblib, json , ta 
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_parquet(r"C:\Users\Admin\Documents\Fintech\90-Day-Fintech-AI\stage-01-reboot\day-05-ml-logreg\rolling_vol.parquet")

In [3]:
# Filtering Data
aapl = df[df["Name"] == 'AAPL'].copy().sort_values("date")
aapl.set_index("date", inplace = True)

### Feature Engineering Pipeline
- We create lag features (t-1 … t-5), rolling stats, and technical indicators.

In [4]:
def build_features(df_price):
    df = df_price.copy()
    # basic lags
    for lag in range(1, 6):
        df[f"ret_lag{lag}"] = df["ret"].shift(lag)
    # rolling volatility
    df["vol_ma_5"] = df["Volatility_30d"].rolling(5).mean()
    # technicals
    df["rsi_14"] = ta.momentum.RSIIndicator(df["close"], window=14).rsi()
    sma_20 = ta.trend.SMAIndicator(df["close"], window=20).sma_indicator()
    df["sma_ratio"] = df["close"] / sma_20
    bb = ta.volatility.BollingerBands(df["close"], window=20, window_dev=2)
    df["bb_position"] = (df["close"] - bb.bollinger_mavg()) / (bb.bollinger_hband() - bb.bollinger_lband())
    
    # target: next-day direction
    df["target"] = (df["ret"].shift(-1) > 0).astype(int)
    return df.dropna()

features = build_features(aapl)

### Train/Validation Split with Time-Series Cross-Validation
- Traditional random split **leaks future info**.
- Use **TimeSeriesSplit** with 5 folds, each fold expanding the training window.

In [5]:
X = features.drop(columns=["close", "ret", "Volatility_30d", "target","Name"], errors = "ignore")
y = features["target"]
tscv = TimeSeriesSplit(n_splits=5)

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")

scores = []
for train_idx, test_idx in tscv.split(X):
    model = logreg.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    prob = model.predict_proba(X.iloc[test_idx])[:, 1]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y.iloc[test_idx], pred, average="binary"
    )
    brier = brier_score_loss(y.iloc[test_idx], prob)
    scores.append({"precision": prec, "recall": rec, "f1": f1, "brier": brier})

metrics = {k: np.mean([s[k] for s in scores]) for k in scores[0]}
print(metrics)

{'precision': np.float64(0.5045751633986928), 'recall': np.float64(0.4239789605416061), 'f1': np.float64(0.30967232386520094), 'brier': np.float64(0.2525185691282666)}


###  Model Evaluation Metrics

- **Precision (0.5046)** → About **50% of the time** when the model predicts a "positive" (e.g., stock going up), it’s correct.  
- **Recall (0.4240)** → Captures only **~42% of actual positives**, so it misses quite a few true cases.  
- **F1 Score (0.3097)** → Weak balance between precision and recall, since both are middling.  
- **Brier Score (0.2525)** → Measures how well-calibrated probabilities are. Closer to 0 is better; **0.25 is not great**, suggesting predicted probabilities aren’t strongly aligned with reality.  

---

###  Interpretation
- The model has slightly better-than-random **precision** (random guess ~0.5 for binary).  
- **Recall is low**, meaning it struggles to capture true positive cases.  
- The **F1 score is low**, showing the balance between precision and recall is weak.  
- The **Brier score** confirms probability estimates aren’t reliable enough yet.  

---



In [6]:
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

### Full-Fit & Model Export

In [7]:
final_model = logreg.fit(X, y)
joblib.dump(final_model, "model.joblib")

['model.joblib']

### Feature Importance
- Logistic regression gives **coefficients**:

In [8]:
coef = pd.Series(final_model.coef_[0], index=X.columns).sort_values(ascending=False)
print(coef.head())

volume         2.237647e-10
bb_position    7.849638e-18
vol_ma_5       3.033550e-18
ret_lag3       6.175045e-19
ret_lag2       4.130554e-19
dtype: float64


### Feature Importance from Logistic Regression

The logistic regression model provides feature importance through the magnitude of its coefficients. Larger absolute values of coefficients suggest stronger influence on the model’s prediction. However, in this case, most coefficients are extremely close to zero (on the order of `1e-18` to `1e-10`), which indicates **very weak predictive power** of the features in their current form.

- **`volume` (2.23e-10)**  
  This has the largest coefficient among the listed features, but the value is still extremely close to zero. This suggests that while trading volume has some influence, it is not strongly predictive in the current model.

- **`bb_position` (7.85e-18)**  
  The Bollinger Band position appears almost negligible. This may indicate that it is not providing meaningful separation between classes (buy/sell signals).

- **`vol_ma_5` (3.03e-18)**  
  The 5-day moving average of volume similarly has minimal impact.

- **`ret_lag3` (6.18e-19)** and **`ret_lag2` (4.13e-19)**  
  Past returns (lagged values) also show extremely small coefficients, suggesting weak predictive contribution.

### Key Insights
- The **tiny coefficients** imply the features, as currently engineered, do not carry strong predictive information for the logistic regression model.  
- This can be due to:
  - High noise in financial time series data.  
  - Nonlinear relationships that logistic regression cannot capture.  
  - Features being weakly correlated with the target variable.  
